In [ ]:
import pandas as pd 
import os 


In [2]:
os.getcwd()
parent_path=os.path.abspath(os.path.join(os.getcwd(),".."))
file_name = "orders_10m.csv"
base_path=os.path.join(parent_path , "otherfile")
file_path= os.path.join(base_path, file_name)
print(file_path)

orders_df  =pd.read_csv(file_path)
orders_df.head()




d:\Project\KrishnaRAGUdemy\otherfile\orders_10m.csv


,order_id,customer_id,product_id,price,order_date,order_status,state,quantity
0,1,124550,38,134.37,2020-03-23,PLACED,Haryana,2
1,2,32801,40,106.00,2009-04-20,RETURNED,Tamil Nadu,1
2,3,467993,48,102.51,2003-02-06,PLACED,Chhattisgarh,3
3,4,620337,40,106.50,2019-05-07,PLACED,Punjab,1
4,5,653851,36,105.54,2002-09-15,PLACED,Rajasthan,3


In [3]:
data = {
    "name" : ["A","b","C","D"] , 
    "roll_no" : [1,2,3,4] , 
    "category" : ["products","seminar", "cartoon","films"]
    
}

df = pd.DataFrame(data)
df.head()

,name,roll_no,category
0,A,1,products
1,b,2,seminar
2,C,3,cartoon
3,D,4,films


In [4]:
with pd.ExcelWriter(f'{base_path}/newfile.xlsx',engine='xlsxwriter') as writer:
    df.to_excel(writer,sheet_name="orders",index=False)

    workbook = writer.book
    workbook.set_properties( { "Title": "orders", 
                              "Subject": "vineet orders data", 
                              "Owner": "vineet", 
                              "Description": "this is orders data" ,
                              "last_modified_by": "vineet"})
    




In [5]:
with pd.ExcelWriter(f'{base_path}/orders.xlsx',engine='xlsxwriter') as writer:
    orders_df.iloc[0:100_00,:].to_excel(writer,sheet_name="orders",index=False)

    




### Baad mei kabhi kaam ayega how to read metadata forthe file

In [6]:
from openpyxl import load_workbook
wb = load_workbook(f'{base_path}/newfile.xlsx')
props = wb.properties

print("Title:", props.title)
print("Subject:", props.subject)
print("Description:", props.description)
print("Owner:", props.creator)
print("Last Modified By:", props.lastModifiedBy)

Title: None
Subject: None
Description: None
Owner: None
Last Modified By: None


### Excel and CSV Parsing Techinques

In [7]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

C:\Users\vinee\AppData\Local\Temp\ipykernel_1360\444435838.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader
d:\Project\KrishnaRAGUdemy\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
unstructuredcsv = UnstructuredCSVLoader(file_path=file_path,mode="elements")
documents =unstructuredcsv.load()
print(f'Metadata infromation >>>>>')


for i , doc in enumerate(documents):
    print(f"\n Element {i+1}")
    print(f'Content :  {doc.page_content[:20]}')
    print(f'Metadata : doc.metadata')
    

d:\Project\KrishnaRAGUdemy\rag\Lib\site-packages\unstructured\partition\csv.py:65: DtypeWarning: Columns (0: 0, 1: 1, 2: 2, 3: 3, 4: 7) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv(file, **read_kw)


XMLSyntaxError: switching encoding: encoder error, line 1, column 1 (<string>, line 1)

In [ ]:
csvobject  = CSVLoader(file_path)
documents =csvobject.load()
print(f'No of documents loaded {len(documents)}')
print(f'Content {documents[0].page_content[:40]}')

### UserDefined advanced CSV or Excel Parsing 

In [15]:
from typing import List
from langchain_core.documents  import Document
def process_excel_advance_way(file_path:str)->List[Document] :
    documents=[]
    # Read all the sheets
    excel_file=pd.ExcelFile(file_path,engine="openpyxl")
    print(excel_file.sheet_names)

    for sheet  in excel_file.sheet_names:
        df = pd.read_excel(file_path,sheet_name=sheet )
        df.head()
        content = f'Rows : {len(df)}'
        content +=f'Columns : {df.columns}'
        content += df.to_string()
        content +=' Other infromation  {df.info()}'

        metadata = { "source" : f"{file_path}" , 
                    "sheet_name" :f'{sheet}' , 
                    "no of rows" : f"{len(df)}", 
                    "num_columns" : f"{len(df.columns)}"                    
                    }
        doc = Document(metadata=metadata, page_content =content)
        documents.append(doc)

    return documents

In [21]:
base_path= os.path.abspath( os.path.join(os.getcwd() , ".."))
folder_name = "otherfile"
file_name="orders.xlsx"
file_path = os.path.join(base_path , folder_name , file_name)
print(file_path)

d:\Project\KrishnaRAGUdemy\otherfile\orders.xlsx


In [ ]:
try:
    file_name = "orders.xlsx"
    
    documents = process_excel_advance_way(file_path)
    print(len(documents))
    for k , v in documents[0].metadata.items():
        print(f' Key : {k} , Value: {v}')

    print(f'Content: {documents[0].page_content}[:100]')
except Exception as e:
    print(f'Error {e}')


['orders']
1
 Key : source , Value: d:\Project\KrishnaRAGUdemy\otherfile\orders.xlsx
 Key : sheet_name , Value: orders
 Key : no of rows , Value: 10000
 Key : num_columns , Value: 8
Content: Rows : 10000Columns : Index(['order_id', 'customer_id', 'product_id', 'price', 'order_date',
       'order_status', 'state', 'quantity'],
      dtype='str')      order_id  customer_id  product_id   price  order_date order_status             state  quantity
0            1       124550          38  134.37  2020-03-23       PLACED           Haryana         2
1            2        32801          40  106.00  2009-04-20     RETURNED        Tamil Nadu         1
2            3       467993          48  102.51  2003-02-06       PLACED      Chhattisgarh         3
3            4       620337          40  106.50  2019-05-07       PLACED            Punjab         1
4            5       653851          36  105.54  2002-09-15       PLACED         Rajasthan         3
5            6       473391          30  154.42

### This is how you read a csv  this maitnian structured of the file

In [ ]:
import csv

with open("file.csv", "r") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)   # row is a list of columns
